# LLM & RAG Workshop (SUT)

© 2026 Parinya Duangklang — https://github.com/parinyad123/LLM-and-RAG-Workshop-SUT

This notebook (code) is licensed under the **MIT License**. See `LICENSE-CODE`.  
Workshop slides are licensed under **CC BY-NC-SA 4.0** (see `LICENSE-CONTENT`).  
You may reuse and adapt with credit; not for commercial use without permission.

In [1]:
!pip install -q \
    langchain-community \
    langchain-core \
    langchain-groq \
    chromadb \
    faiss-cpu \
    sentence-transformers \
    gradio \
    pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/1

## 0.2 Import Libraries

In [2]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader, PyPDFDirectoryLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_community.vectorstores import Chroma, FAISS

from langchain_groq import ChatGroq

import os
import time
import numpy as np
from pathlib import Path
import re

from google.colab import userdata, files

import gradio as gr

## 0.3 Configure API keys and initialize LLM

In [3]:
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    llm = ChatGroq(
        model = "openai/gpt-oss-120b",
        # model="llama-3.3-70b-versatile"
        temperature=0.2
    )
    print("Using Groq")
except:
    print("Groq key not found!")
    print("Please add GROQ_API_KEY to Colab Secrets\n")

Using Groq


## 0.4 Initialize Embeddings



In [4]:
embeddings = HuggingFaceEmbeddings(
    # model_name="sentence-transformers/all-mpnet-base-v2",
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print(f"Model loaded (384 dimensions)")

print("\nTesting embeddings...")

# Convert a sample text string into a Vector (a long sequence of numerical data)
test_vec = embeddings.embed_query(
    "SpaceX, founded in 2002, is worth $800 billion based on a private tender offer in December 2025"
)

# Verify system functionality by checking the Vector length
print(f"Embeddings working! (dim={len(test_vec)})")

/tmp/ipykernel_3195/161620214.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded (384 dimensions)

Testing embeddings...
Embeddings working! (dim=384)


## 4.3 Document Formatter

In [5]:
def format_docs(docs):
    formatted = []

    # Iterate through each document to extract content and metadata
    for i, doc in enumerate(docs, 1):
        page   = doc.metadata.get('page', '?')
        raw    = doc.metadata.get('source', '')
        source = os.path.basename(raw) if raw else 'Unknown'

        # Construct the formatted string with clear identification
        formatted.append(
            f"{source} | Page {page}]\n{doc.page_content.replace('\n', ' ')}"
        )

    # Join all formatted documents with a double newline separator
    result = "\n\n".join(formatted)

    return result

In [6]:
def clean_document(docs: list) -> list:

    cleaned_docs = []

    for doc in docs:
        text = doc.page_content

        # 1. Remove Null bytes (binary artifacts often found in PDFs/scans)
        text = text.replace('\x00', '')

        # 2. Collapse large clusters of spaces (3+) into a single space
        text = re.sub(r' {3,}', ' ', text)

        # 3. Normalize vertical whitespace: Reduce excessive newlines (3+) to double newlines
        text = re.sub(r'\n{3,}', '\n\n', text)

        # 4. Convert single line-wraps (\n) into spaces
        text = text.replace('\n\n', '<<PARA>>')
        text = text.replace('\n', ' ')
        text = text.replace('<<PARA>>', '\n\n')

        # 5. Clean up any double spaces that may have been created during conversion
        text = re.sub(r' {2,}', ' ', text)

        # Reconstruct Document with sanitized content and original metadata
        cleaned_docs.append(
            Document(
                page_content=text,
                metadata=doc.metadata
            )
        )

    return cleaned_docs

# Part 6: BYOD (Bring Your Own Document) Practice

## 6.1 Document Loader — รองรับหลาย Format

In [7]:
def load_document(file_path):

    # Extract the file extension and convert to lowercase
    ext = os.path.splitext(file_path)[-1].lower()

    # Map extensions to their respective LangChain loader classes
    loaders = {
        ".pdf" : PyPDFLoader,
        ".txt" : TextLoader,
        ".docx": Docx2txtLoader,
    }

    # Check if the extension is supported; if not, raise an error
    if ext not in loaders:
        raise ValueError(f"❌ Unsupported format: {ext}\n   Supported: {list(loaders.keys())}")

    # Initialize the loader and load the document data
    loader = loaders[ext](file_path)
    docs   = loader.load()

    return docs

## 6.2 Document Inspector

## 6.3 Smart Chunking — ปรับตามประเภทเอกสาร

In [8]:
def chunk_document(docs, doc_type="general"):

    # Configuration presets for different chunking strategies
    configs = {
        "c300_o150":  {"chunk_size": 300,  "chunk_overlap": 150},
        "c500_o250":  {"chunk_size": 500,  "chunk_overlap": 250},
        "c700_o350":  {"chunk_size": 700,  "chunk_overlap": 350},
        "c1000_o500": {"chunk_size": 1000, "chunk_overlap": 500},
    }

    # Select config based on doc_type, default to c500_o250 if not found
    config  = configs.get(doc_type, configs["c500_o250"])

    # Initialize the RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = config["chunk_size"],    # Maximum characters per chunk
        chunk_overlap = config["chunk_overlap"], # Overlapping characters between chunks
        separators = ["\n\n", "\n", ". ", " ", ""] # Priority list of delimiters
    )

    # Split the documents into smaller chunks
    chunks = splitter.split_documents(docs)

    return chunks

## 6.4 Build BYOD VectorStore

In [9]:
def build_vectorstore(chunks, collection_name="byod_collection"):

    # Start timer to measure indexing performance
    start = time.time()

    # Initialize Chroma vectorstore from the provided document chunks
    vs = Chroma.from_documents(
        documents = chunks,
        embedding = embeddings,
        collection_name = collection_name
    )

    elapsed = time.time() - start
    print(f"🚀 Vectorstore built in: {elapsed:.2f} seconds")

    return vs

## 6.5 Build BYOD RAG Chain

In [10]:
def build_byod_rag(vectorstore, doc_description="the provided document"):


    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 5,
            "fetch_k": 20,
            "lambda_mult": 0.5
        }
    )

    # Define the System Prompt for the LLM
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""You are a helpful assistant answering questions about {doc_description}.

          Instructions:
          - Answer based ONLY on the provided context (ตอบคำถามจากบริบทที่ให้มาเท่านั้น)
          - Be concise and accurate (ตอบให้กระชับและแม่นยำ)
          - If the answer is not in the context, say "I don't have enough information"
          - When referencing information, cite using exact format: [filename | Page X]

          Context:
          {{context}}"""),
                  ("user", "{question}")
    ])

    # Construct the LCEL Chain
    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return chain, retriever

## 6.6 Gradio BYOD Interface — Full Workshop App

In [11]:
import gradio as gr
import tempfile

# Global variables to store the RAG components for the current session
current_chain = None
current_retriever = None

def process_document(file, doc_type, doc_description):

    global current_chain, current_retriever

    if file is None:
        return "❌ Please upload file before"

    try:
        log = ""

        # Step 1: Load
        docs = load_document(file.name)
        log += f"✅ Loaded: {os.path.basename(file.name)}\n"
        log += f"   {len(docs)} pages\n\n"

        # Step 2: Inspect (Document Health Check)
        log += "📊 Document Health Check\n"
        log += "="*40 + "\n"

        full_text     = " ".join([d.page_content for d in docs])
        total_chars   = len(full_text)
        null_bytes    = full_text.count('\x00')
        excess_spaces = len(re.findall(r' {3,}', full_text))
        excess_lines  = len(re.findall(r'\n{3,}', full_text))

        log += f"Pages           : {len(docs)}\n"
        log += f"Total chars     : {total_chars:,}\n\n"
        log += f"Null bytes      : {null_bytes}  {'⚠️ Clean needed' if null_bytes > 0 else '✅'}\n"
        log += f"Excess spaces   : {excess_spaces}  {'⚠️ Clean needed' if excess_spaces > 0 else '✅'}\n"
        log += f"Excess newlines : {excess_lines}  {'⚠️ Clean needed' if excess_lines > 0 else '✅'}\n\n"

        # Step 3: Clean (If issues found)
        issues = any([null_bytes > 0, excess_spaces > 0, excess_lines > 0])
        if issues:
            log += "\n⚙️  Cleaning document...\n"
            docs = clean_document(docs) # Assume clean_document is defined
            log += "✅ Cleaning complete\n\n"
        else:
            log += "\n✅ No cleaning needed\n\n"

        # Step 4: Chunk
        chunks = chunk_document(docs, doc_type)
        log += f"✅ Chunked: {len(chunks)} chunks (type: {doc_type})\n\n"

        # Step 5: Vectorstore
        vs  = build_vectorstore(chunks, "byod_doc")
        log += f"✅ Vectorstore: {vs._collection.count()} vectors\n\n"

        # Step 6: RAG Chain
        current_chain, current_retriever = build_byod_rag(vs, doc_description)
        log += "🚀 Ready! ไปที่ Chat tab ได้เลย"

        return log

    except Exception as e:
        return f"❌ Error: {str(e)}"


def chat_byod(message, history):
    """
    Handles the chat interface logic.
    จัดการส่วนการรับข้อความและตอบกลับใน Chat
    """
    global current_chain, current_retriever

    if current_chain is None:
        history.append([message, "❌ Please upload document into Setup tab"])
        return "", history

    if not message.strip():
        return "", history

    # Generate answer from the chain
    docs    = current_retriever.invoke(message)
    answer  = current_chain.invoke(message)

    # Append source information for transparency
    sources = f"\n\n---\n📚 Sources: "
    pages   = [str(d.metadata.get('page','?')) for d in docs]
    sources += f"Page {', '.join(sorted(set(pages)))}"

    history.append([message, answer])
    return "", history


# Build Gradio App
# UI Layout using gr.Blocks
with gr.Blocks(title="📚 BYOD RAG Workshop") as demo_byod:

    gr.Markdown("# 📚 BYOD RAG — Bring Your Own Document")

    with gr.Tabs():
        # Setup Tab for file processing
        with gr.Tab("⚙️ Setup"):
            gr.Markdown("### Step 1: Upload your document")

            with gr.Row():
                with gr.Column():
                    file_input = gr.File(label="Upload Document", file_types=[".pdf", ".txt", ".docx"])
                    doc_type = gr.Radio(
                        choices=["c300_o150", "c500_o250", "c700_o350", "c1000_o500"],
                        value="c500_o250",
                        label="Chunk Size (chunk_size / overlap)"
                    )
                    doc_desc = gr.Textbox(label="Document Description", value="the uploaded document")
                    upload_btn = gr.Button("🚀 Process Document", variant="primary")

                with gr.Column():
                    process_log = gr.Textbox(label="Processing Log", lines=20)

            upload_btn.click(process_document, inputs=[file_input, doc_type, doc_desc], outputs=[process_log])

        # Chat Tab for interaction
        with gr.Tab("💬 Chat"):
            chatbot = gr.Chatbot(height=450)
            msg_box = gr.Textbox(placeholder="Ask about your document...")
            with gr.Row():
                submit_btn = gr.Button("Send", variant="primary")
                clear_btn  = gr.Button("Clear")

            submit_btn.click(chat_byod, [msg_box, chatbot], [msg_box, chatbot])
            msg_box.submit(chat_byod, [msg_box, chatbot], [msg_box, chatbot])
            clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg_box])

# Launch the app with a public share link
demo_byod.launch(share=True)

/tmp/ipykernel_3195/3881953625.py:122: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=450)
/tmp/ipykernel_3195/3881953625.py:122: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=450)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d57f46b91d09eb0b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
